In [42]:
# import packages
import pandas as pd
import numpy as np
import re

In [43]:
#create dataframes

course_path = '/Users/christiangroeger/Desktop/KI-Campus/yuzhi/Course - Category - Enrollments.csv' 
cert_path = '/Users/christiangroeger/Desktop/KI-Campus/yuzhi/Course - Certificates.csv'

course_df = pd.read_csv(course_path)
cert_info = pd.read_csv(cert_path)
#course_df.head()
cert_info.head()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/christiangroeger/Desktop/KI-Campus/yuzhi/Course - Category - Enrollments.csv'

In [ ]:
# groupby cert_info by courseID
participation_key = r'teilnahmebestätigung|participation|teilnahmebescheinigung'
achievement_key = r'leistungsnachweis|leistungsnachweise|achivement|record of achievement|zertifikat|certificate|transcript of records'

cert_info['name_lower'] = cert_info['certificateName'].str.lower()

def classify_cert(name):
    if pd.isna(name):
        return None
    name = name.lower()
    if re.search(participation_key, name):
        return 'CoP'
    if re.search(achievement_key, name):
        return 'RoA'
    else:
        return None

cert_info['cert_type'] = cert_info['name_lower'].apply(classify_cert)

cert_info_filtered = cert_info[cert_info['cert_type'].notna()]

cert_df = (cert_info_filtered.groupby(['courseID', 'cert_type'])['amount'].sum().unstack('cert_type').reset_index())

cert_df.fillna(0, inplace = True)

cert_df[['CoP', 'RoA']] = cert_df[['CoP', 'RoA']].astype(int)

cert_df.columns = ['courseID', 'comfirmation_of_participation', 'records_of_achievement']

# renaming columns
cert_df.columns = [
    'courseID',
    'Acquired # Comfirmation of Participations',
    'Acquired # Records of Achievement'
]


print(cert_df.shape[0])

cert_df.head()


In [ ]:
# merge two dataframes
course_df = course_df[['courseID', 'courseName', 'categoryName', 'enrollments']]

# renaming columns
course_df = course_df.rename(columns={'enrollments': 'Enrolled learners'})

print(len(course_df))
print(len(cert_df))

merged_df = course_df.merge(cert_df, on = 'courseID', how = 'left')

print(len(merged_df))

merged_df.head()
merged_df.columns

In [ ]:
# import base table (with info: which courses live)
base_table_path = '/Users/christiangroeger/Desktop/KI-Campus/yuzhi/KIC-course completion rate.csv'

base_table = pd.read_csv(base_table_path, sep = ';', encoding = 'utf8')

base_table.head()
print(len(base_table))
base_table.columns

In [ ]:
# merge the cert data with the base table
base_merge = merged_df.merge(base_table, on = 'courseID', how = 'outer')
base_merge 


In [ ]:
# --- Step 1: create extended table ---
base_new_columns = base_merge.copy()

cop = pd.to_numeric(base_new_columns["Acquired # Comfirmation of Participations"], errors="coerce")
roa = pd.to_numeric(base_new_columns["Acquired # Records of Achievement"], errors="coerce")
enr = pd.to_numeric(base_new_columns["Enrolled learners"], errors="coerce")

enr_safe = enr.replace(0, np.nan)

base_new_columns["Rate CoP"] = cop / enr_safe
base_new_columns["Rate RoA (success)"] = roa / enr_safe

# --- Step 2: remove duplicates if they exist ---
base_new_columns = base_new_columns.loc[:, ~base_new_columns.columns.duplicated()]

# --- Step 3: reorder so new columns are after index 6 ---
cols = list(base_new_columns.columns)
insert_at = 5

# Remove them from cols if they are already at the end
cols = [c for c in cols if c not in ["Rate CoP", "Rate RoA (success)"]]

# Now insert them after index 6
new_order = cols[:insert_at+1] + ["Rate CoP", "Rate RoA (success)"] + cols[insert_at+1:]

base_new_columns = base_new_columns[new_order]
base_new_columns.head()

In [ ]:
# export data for spreadsheet check
# outcome: list of all courses
# merged_df.to_excel('C:/Users/ywa/Documents/Data/Moodle Query/June2025/MoodleEnrole.xlsx', index = False)
base_new_columns.to_excel('/Users/christiangroeger/Desktop/KI-Campus/yuzhi/MoodleEnrole.xlsx', index = False)

In [ ]:
# extract live courses and divide in two lists: non-microdegrees and microdegrees for easy copy-paste

# list of live courses without microdegrees: UPDATE EVERY TIME by copying the CourseID column!
list1 = [
    360,337,318,313,291,285,282,279,274,268,259,255,252,251,250,248,245,
243,242,241,237,236,234,231,229,228,224,223,221,202,199,197,187,177,
176,164,141,140,136,127,121,111,106,99,95,76,75,74,64,63,58,29,28,27,19
]

# list of live microdegree courses: UPDATE EVERY TIME by copying the CourseID column!
list2 = [332, 331, 330, 329, 323, 322, 362, 145, 144, 143, 142, 238, 192, 191, 266, 151, 103, 100, 109, 214, 6, 67, 66, 213, 56, 80, 10, 215]


# Filter df for each list
df_list1 = base_new_columns[base_new_columns['courseID'].isin(list1)]
df_list2 = base_new_columns[base_new_columns['courseID'].isin(list2)]

# For list2, preserve exact order given
df_list2 = pd.Categorical(df_list2['courseID'], categories=list2, ordered=True)
df_list2 = base_new_columns[base_new_columns['courseID'].isin(list2)].sort_values(by='courseID', key=lambda col: col.map({v:i for i,v in enumerate(list2)}))

# Save to Excel or CSV if needed
df_list1.to_excel("outcome_list1.xlsx", index=False)
df_list2.to_excel("outcome_list2.xlsx", index=False)
